In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

print(torch.__version__)

2.12.1+cpu
Im ready to use PyTorch!


In [ ]:
D_true = 1.0 # true diffusivity
T = 1.0 # final time

# for context, the rule that the solution must obey is given by u_t = D*u_xx
# this is how we get the pinn residual (by derivating)

# u_t represents du/dt, or the rate the temperature is changing over time at a point

# D = k/(p * c_p) = thermal conductivity / (density * specific heat capacity) = how easily heat flows / how hard it is to heat a material up
# that is thermal diffusivity, which is how quickly heat diffuses through a material

# u_xx is shorthand for d^2u/dx^2, or the curvature of the temperature distribution in space at a point

# we are using diffusivity * u_xx in this case, it is the equivalent to flux_x / (p * c_p)

def analytical_solution(x,t):
    # just gives temp at position x and time t AKA actual temp function u(x,t)
    return torch.sin(np.pi *x) + torch.exp(-D_true * (np.pi**2)*t)

# x's and t's written separately for readability
print(analytical_solution(torch.tensor([[0.5]]), torch.tensor([[0.0]])).item()) # -> middle of object, time 0 s
print(analytical_solution(torch.tensor([[0.5]]), torch.tensor([[0.5]])).item()) # -> middle of object, time 0.5 s

2.0
1.0071918964385986


In [4]:
class HeatPINN(nn.Module):
    def __init__(self, n_hidden=32, n_layers=4): # 32 neurons each hidden layer, 4 layers
        super().__init__()
        layers = [nn.Linear(2, n_hidden), nn.Tanh()]
        for _ in range(n_layers -1):
            layers += [nn.Linear(n_hidden, n_hidden), nn.Tanh()]

        # now layers contains a list of network steps

        layers += [nn.Linear(n_hidden, 1)] # output step, gives temp at a point
        self.net = nn.Sequential(*layers) # just makes the pipeline of layers + unpacks list

    def forward(self, x, t):
        # concatenates x and t into a single input vector for the network
        # nn expects 2 input features per/row 
        return self.net(torch.cat([x, t], dim=1))

In [5]:
def heat_residual(model, x, t, D):
    # clone + track derivatives
    x = x.clone().requires_grad_(True)
    t = t.clone().requires_grad_(True)
    
    u = model(x, t) # sends x and t to the network  

    # create_graph = True in case we want to compute higher order ders later on
    u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0] # du/dt
    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0] # du/dx
    u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0] # d^2u/dx^2

    return u_t-D*u_xx